# Experiments

## Setup: Import Libraries and Scripts

In [ ]:
import pandas as pd
from IPython.display import display, HTML
import optimize_prompt as opt  # Full optimization script
import optimize_prompt4 as opt4  # Full optimization script
import zero_shot_baseline as zsb  # Zero-shot baseline script

import pickle
import os
from datetime import datetime

# Style for better table display
pd.set_option('display.max_columns', None)
pd.set_option('display.expand_frame_repr', False)
pd.set_option('display.max_colwidth', None)  # Show full content in cells

# Path for saving experiment runs
RUNS_PICKLE_PATH = 'experiment_runs.pkl'

## Define Experiments

Add/edit experiments here. Each is a dict with:
- `'name'`: A label for the experiment.
- `'script'`: 'optimize' or 'zero_shot'.
- Other keys: Parameters for main() (e.g., generations, model_name).

In [ ]:
experiments = [
      {
        'name': 'Full Optimization - Reasoning Model',
        'script': 'optimize4',
        'generations': 10,
        'pop_size': 8,
        'train_sample_size': 100,
        'test_sample_size': 1000,
        'model_name': 'google/gemini-2.5-flash',
        'use_bandit_instr': True,
        'use_bandit_template': True,
        'statutory_context_enabled': True,
        'contract_context_enabled': True
    },
    {
        'name': 'Zero Shot - Reasoning Model',
        'script': 'zero_shot',
        'test_sample_size': 1000,
        'model_name': 'google/gemini-2.5-flash',
        'statutory_context_enabled': False,
        'contract_context_enabled': False
    },
]

experiments_backlog = [
                          {
        'name': 'Full Optimization',
        'script': 'optimize4',
        'generations': 8,
        'pop_size': 4,
        'train_sample_size': 30,
        'test_sample_size': 300,
        'model_name': 'google/gemini-2.5-flash',
        'use_bandit_instr': True,
        'use_bandit_template': True,
        'statutory_context_enabled': True,
        'contract_context_enabled': True
    },
                  {
        'name': 'Full Optimization w/o Contexts',
        'script': 'optimize4',
        'generations': 8,
        'pop_size': 4,
        'train_sample_size': 30,
        'test_sample_size': 300,
        'model_name': 'google/gemini-2.5-flash',
        'use_bandit_instr': True,
        'use_bandit_template': True,
        'statutory_context_enabled': False,
        'contract_context_enabled': False
    },

            {
        'name': 'Zero Shot',
        'script': 'zero_shot',
        'test_sample_size': 300,
        'model_name': 'google/gemini-2.5-flash',
        'statutory_context_enabled': False,
        'contract_context_enabled': False
    },
        {
        'name': 'Zero Shot w/o Contexts',
        'script': 'zero_shot',
        'test_sample_size': 300,
        'model_name': 'google/gemini-2.5-flash',
        'statutory_context_enabled': False,
        'contract_context_enabled': False
    },
                  {
        'name': 'Full Optimization',
        'script': 'optimize4',
        'generations': 15,
        'pop_size': 6,
        'train_sample_size': 10,
        'test_sample_size': 300,
        'model_name': 'openrouter/horizon-beta',
        'use_bandit_instr': True,
        'use_bandit_template': True,
        'statutory_context_enabled': True,
        'contract_context_enabled': True
    },
                  {
        'name': 'Full Optimization w/o Contexts',
        'script': 'optimize4',
        'generations': 15,
        'pop_size': 6,
        'train_sample_size': 10,
        'test_sample_size': 300,
        'model_name': 'openrouter/horizon-beta',
        'use_bandit_instr': True,
        'use_bandit_template': True,
        'statutory_context_enabled': False,
        'contract_context_enabled': False
    },
      {
        'name': 'Full Optimization - w META INSTRUCTION 3 & META TEMPLATE 3 & INSTR_STRATEGIES_ORIGINAL w/o early stopping',
        'script': 'optimize',
        'generations': 40,
        'pop_size': 12,
        'train_sample_size': 20,
        'test_sample_size': 1000,
        'model_name': 'google/gemini-2.5-flash',
        'use_bandit_instr': True,
        'use_bandit_template': True,
        'statutory_context_enabled': True,
        'contract_context_enabled': True
    },
    {
        'name': 'Full Optimization - w META INSTRUCTION 3 & META TEMPLATE 3 & INSTR_STRATEGIES_ORIGINAL w/o early stopping',
        'script': 'zero_shot',
        'test_sample_size': 1000,
        'model_name': 'google/gemini-2.5-flash',
        'statutory_context_enabled': True,
        'contract_context_enabled': True
    },
        {
        'name': 'Full Optimization - w META INSTRUCTION 3 & META TEMPLATE 3 & INSTR_STRATEGIES_ORIGINAL w/o early stopping - No Bandit',
        'script': 'optimize',
        'generations': 20,
        'pop_size': 4,
        'train_sample_size': 10,
        'test_sample_size': 300,
        'model_name': 'google/gemini-2.5-flash-lite-preview-06-17',
        'use_bandit_instr': False,
        'use_bandit_template': False,
        'statutory_context_enabled': True,
        'contract_context_enabled': True
    },
        {
        'name': 'Full Optimization - w META INSTRUCTION 3 & META TEMPLATE 3 - No Bandit',
        'script': 'optimize',
        'generations': 40,
        'pop_size': 4,
        'train_sample_size': 10,
        'test_sample_size': 300,
        'model_name': 'google/gemini-2.5-flash-lite-preview-06-17',
        'use_bandit_instr': False,
        'use_bandit_template': False,
        'statutory_context_enabled': True,
        'contract_context_enabled': True
    },
]

## Run Experiments

This cell runs each experiment and collects results, saving them to a pickle file with a timecode.

In [ ]:
results = []

for exp in experiments:
    print(f"\n=== Running Experiment: {exp['name']} ===")
    script = exp.pop('script')  # Remove script key for passing to main
    name = exp.pop('name')  # Remove name for passing to main
    run_time = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    
    try:
        if script == 'optimize':
            result = opt.main(**exp)
        elif script == 'optimize4':
            result = opt4.main(**exp)
        elif script == 'zero_shot':
            result = zsb.main(**exp)
        else:
            raise ValueError(f"Unknown script: {script}")
        
        # Flatten metrics for table
        metrics = result['test_metrics']
        flat_result = {
            'Experiment Name': name,
            'Script': script,
            **exp,  # Add back parameters
            'Best Instruction': result['best_instruction'],
            'Best Template': result['best_template'],
            'Sample Size': metrics['sample_size'],
            'Valid Predictions': metrics['valid_predictions'],
            'Total Predictions': metrics['total_predictions'],
            'Accuracy': metrics['accuracy'],
            'Precision': metrics['precision'],
            'Recall': metrics['recall'],
            'F1 Micro': metrics['f1_micro'],
            'F1 Macro': metrics['f1_macro'],
            'Adjusted F1 Macro': metrics['adjusted_f1_macro'],
            'Support (0/1)': f"{metrics['support'].get('0', 0)} / {metrics['support'].get('1', 0)}",
            'Unique y_true': ', '.join(metrics['unique_y_true']),
            'Unique y_pred': ', '.join(metrics['unique_y_pred']),
            'Detailed Report': metrics['detailed_report_string'],  # Full string for details
            'Full Classification Report (Dict)': metrics['classification_report'],  # Raw dict if needed
            'Run Time': run_time
        }
        results.append(flat_result)
    except Exception as e:
        print(f"Error in experiment '{name}': {e}")
        results.append({'Experiment Name': name, 'Error': str(e), 'Run Time': run_time})

# Load previous runs if exists
if os.path.exists(RUNS_PICKLE_PATH):
    with open(RUNS_PICKLE_PATH, 'rb') as f:
        past_runs = pickle.load(f)
else:
    past_runs = []

# Add new results to past runs and save
all_runs = past_runs + results
with open(RUNS_PICKLE_PATH, 'wb') as f:
    pickle.dump(all_runs, f)


=== Running Experiment: Full Optimization - Reasoning Model ===
============ Generation 1 ============


Evaluating population:   0%|          | 0/4 [00:00<?, ?it/s]

---- Sent in Batch 1 ----
Instruction: Classify the following clause from a Terms of Service contract as fair (0) or unfair (1) using the statutory context and contract context for better understanding. Respond only with '0' or '1'.
Clause: 3.1 these terms and conditions shall apply to all orders and contracts made or to be made by us for the sale and supply of products .
Statutory Context: According to art. 3 of the Directive 93/13 on Unfair Terms in Consumer Contracts, a contractual term is unfair if: 1) it has not been individually negotiated; and 2) contrary to the requirement of good faith, it causes a significant imbalance in the parties' rights and obligations, to the detriment of the consumer. This general definition is further specified in the Annex to the Directive, containing an indicative and non-exhaustive list of the terms which may be regarded as unfair, as well as in a few dozen judgments of the Court of Justice of the EU. Examples of unfair clauses encompass taking jur

Evaluating population:  25%|██▌       | 1/4 [02:04<06:14, 124.72s/it]

⭐ Adjusted F1 Macro Score: 0.6915
---- Sent in Batch 1 ----
Instruction: Classify the following clause from a Terms of Service contract as fair (0) or unfair (1) using the statutory context and contract context for better understanding. Respond only with '0' or '1'.
Clause: you acknowledge that the expedia companies pre-negotiate certain room rates with hotel suppliers to facilitate the booking of reservations .
Statutory Context: According to art. 3 of the Directive 93/13 on Unfair Terms in Consumer Contracts, a contractual term is unfair if: 1) it has not been individually negotiated; and 2) contrary to the requirement of good faith, it causes a significant imbalance in the parties' rights and obligations, to the detriment of the consumer. This general definition is further specified in the Annex to the Directive, containing an indicative and non-exhaustive list of the terms which may be regarded as unfair, as well as in a few dozen judgments of the Court of Justice of the EU. Exampl

## Display Results Table

Interactive table with all parameters and metrics. Sorted by most recent run.

In [ ]:
# Load all runs from pickle and display sorted by most recent run time
import pickle
import pandas as pd
from IPython.display import display, HTML

RUNS_PICKLE_PATH = 'experiment_runs.pkl'

if os.path.exists(RUNS_PICKLE_PATH):
    with open(RUNS_PICKLE_PATH, 'rb') as f:
        all_runs = pickle.load(f)
    # Sort by 'Run Time' descending
    all_runs_sorted = sorted(all_runs, key=lambda x: x.get('Run Time', ''), reverse=True)
    df_results = pd.DataFrame(all_runs_sorted)
    if not df_results.empty:
        styled_df = df_results.style.set_properties(**{'text-align': 'left', 'white-space': 'pre-wrap'}).set_table_styles([
            {'selector': 'th', 'props': [('text-align', 'left')]}
        ]).background_gradient(cmap='viridis', subset=['Adjusted F1 Macro'])
        display(HTML("<h3>All Experiment Runs (Most Recent First)</h3>"))
        display(styled_df)
    else:
        print("No results to display.")
else:
    print("No experiment runs found.")

,Experiment Name,Error,Run Time,Script,generations,pop_size,train_sample_size,test_sample_size,model_name,use_bandit_instr,use_bandit_template,statutory_context_enabled,contract_context_enabled,Best Instruction,Best Template,Sample Size,Valid Predictions,Total Predictions,Accuracy,Precision,Recall,F1 Micro,F1 Macro,Adjusted F1 Macro,Support (0/1),Unique y_true,Unique y_pred,Detailed Report,Full Classification Report (Dict)
0,Zero Shot w/o Contexts,'adjusted_f1_macro',2025-08-04 10:07:43,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
1,Zero Shot,'adjusted_f1_macro',2025-08-04 10:03:16,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
2,Zero Shot,main() got an unexpected keyword argument 'use_bandit_instr',2025-08-04 09:57:36,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
3,Zero Shot w/o Contexts,main() got an unexpected keyword argument 'use_bandit_instr',2025-08-04 09:57:36,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan,nan
4,Full Optimization w/o Contexts,nan,2025-08-04 09:30:00,optimize4,15.000000,6.000000,10.000000,300.000000,google/gemini-2.5-flash,True,True,False,False,"As a specialized legal reasoning engine for consumer contract fairness, your objective is to classify individual clauses as either '0' (Fair) or '1' (Unfair) with absolute precision. **Unfairness Definition:** A clause is **unfair (1)** if it violates the principle of good faith by creating a significant imbalance in contractual rights and obligations to the consumer's detriment. This specifically includes clauses that: * **Significantly limit the service provider's liability** or grant the provider disproportionate discretion without equivalent consumer safeguards. * **Impose excessive or disproportionate burdens on the consumer** relative to the service provided. * **Grant the service provider unilateral rights** to modify terms, terminate the contract, or interpret clauses without adequate notice, justification, or reciprocal consumer protections. **Fairness Definition:** A clause is **fair (0)** if it does not meet the criteria for unfairness, maintaining a reasonable and balanced distribution of rights and obligations consistent with good faith principles. To classify, perform the following: 1. **Goal:** Determine if the clause is '0' (Fair) or '1' (Unfair) based on the provided definitions. 2. **Definition Reference:** Recall the precise","You are an expert legal AI specializing in contract law and fairness assessments. Your task is to classify a given clause as either fair or unfair. Respond exclusively with the digit '0' if the clause is fair, or '1' if the clause is unfair. Provide absolutely nothing else in your response. **Instruction:** ** **Clause to classify:** **",300.000000,300.000000,300.000000,0.733333,0.193182,0.653846,0.733333,0.566818,0.566818,274.0 / 26.0,"0, 1","0, 1",precision recall f1-score support 0 0.9575 0.7409 0.8354 274 1 0.1932 0.6538 0.2982 26 accuracy 0.7333 300 macro avg 0.5754 0.6974 0.5668 300 weighted avg 0.8913 0.7333 0.7888 300,"{'0': {'precision': 0.9575471698113207, 'recall': 0.7408759124087592, 'f1-score': 0.8353909465020576, 'support': 274.0}, '1': {'precision': 0.19318181818181818, 'recall': 0.6538461538461539, 'f1-score': 0.2982456140350877, 'support': 26.0}, 'accuracy': 0.7333333333333333, 'macro avg': {'precision': 0.5753644939965694, 'recall': 0.6973610331274565, 'f1-score': 0.5668182802685726, 'support': 300.0}, 'weighted avg': {'precision': 0.8913021726700971, 'recall': 0.7333333333333333, 'f1-score': 0.7888383510215868, 'support': 300.0}}"
5,Full Optimization,nan,2025-08-04 09:02:16,optimize4,15.000000,6.000000,10.000000,300.000000,google/gemini-2.5-flash,True,True,True,True,"Classify the contract clause fairness: '0' for fair, '1' for unfair.",``` ***STATUTORY CONTEXT*** ***CONTRACT CONT